# 1. Setup

In [1]:
import pandas as pd 

In [2]:
cols = ["id","entity","sentiment","text"]
train = pd.read_csv("../data/raw/twitter_training.csv",header = None,names = cols)
validation = pd.read_csv("../data/raw/twitter_validation.csv",header= None,names= cols)

# 2. Cleaning

In [3]:
train = train.dropna()

In [4]:
train.isna().sum()

id           0
entity       0
sentiment    0
text         0
dtype: int64

In [5]:
validation.isna().sum()

id           0
entity       0
sentiment    0
text         0
dtype: int64

# 3. Duplicates&Leakage

In [6]:
train["text"].duplicated().sum()

np.int64(4505)

In [7]:
labels_per_text = train.groupby("text")["sentiment"].nunique()
conflicting_texts = labels_per_text[labels_per_text > 1].index
print("Conflicting texts:", len(conflicting_texts))

Conflicting texts: 137


In [8]:
print("Before:", train.shape)
train = train[~train["text"].isin(conflicting_texts)]
print("After removing conflicts:", train.shape)

Before: (73996, 4)
After removing conflicts: (71876, 4)


In [9]:
train = train.drop_duplicates(subset="text")
print("After removing duplicates:", train.shape)

After removing duplicates: (69354, 4)


In [10]:
train["text"].duplicated().sum()

np.int64(0)

In [11]:
leak = validation["text"].isin(train["text"])
print(leak.sum(), "of", len(validation), f"({leak.mean():.1%})")

514 of 1000 (51.4%)


# 4. Split

In [12]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(train, groups=train["id"]))

train_df = train.iloc[train_idx]
test_df = train.iloc[test_idx]

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (55526, 4)
Test: (13828, 4)


In [13]:
test_df["id"].isin(train_df["id"]).sum()

np.int64(0)

In [14]:
pd.DataFrame({
    "train": train_df["sentiment"].value_counts(normalize=True).round(3),
    "test": test_df["sentiment"].value_counts(normalize=True).round(3),
})

,train,test
sentiment,,
Negative,0.307,0.297
Positive,0.272,0.282
Neutral,0.247,0.237
Irrelevant,0.174,0.183


In [15]:
def clean_column(series):
    return (
        series
        .str.lower()
        .str.replace(r"http\S+|www\S+", "", regex=True)   # links
        .str.replace(r"@\w+", "", regex=True)             # @usernames
        .str.replace(r"[^a-z\s]", " ", regex=True)        # keep only letters and spaces
        .str.replace(r"\s+", " ", regex=True)             # multiple spaces → one
        .str.strip()                                      # spaces at start/end
    )

train_df = train_df.copy()
test_df = test_df.copy()

train_df["clean_text"] = clean_column(train_df["text"])
test_df["clean_text"] = clean_column(test_df["text"])
validation["clean_text"] = clean_column(validation["text"])

train_df[["text", "clean_text"]].head(10)

,text,clean_text
0,im getting on borderlands and i will murder yo...,im getting on borderlands and i will murder yo...
1,I am coming to the borders and I will kill you...,i am coming to the borders and i will kill you...
2,im getting on borderlands and i will kill you ...,im getting on borderlands and i will kill you all
3,im coming on borderlands and i will murder you...,im coming on borderlands and i will murder you...
4,im getting on borderlands 2 and i will murder ...,im getting on borderlands and i will murder yo...
5,im getting into borderlands and i can murder y...,im getting into borderlands and i can murder y...
12,"Rock-Hard La Varlope, RARE & POWERFUL, HANDSOM...",rock hard la varlope rare powerful handsome ja...
13,"Rock-Hard La Varlope, RARE & POWERFUL, HANDSOM...",rock hard la varlope rare powerful handsome ja...
14,"Rock-Hard La Varlope, RARE & POWERFUL, HANDSOM...",rock hard la varlope rare powerful handsome ja...
15,"Rock-Hard La Vita, RARE BUT POWERFUL, HANDSOME...",rock hard la vita rare but powerful handsome j...


In [16]:
# 1. drop tweets that became empty after cleaning
print("Empty in train:", (train_df["clean_text"] == "").sum())
print("Empty in test:", (test_df["clean_text"] == "").sum())

train_df = train_df[train_df["clean_text"] != ""]
test_df = test_df[test_df["clean_text"] != ""]

Empty in train: 84
Empty in test: 31
